## 📋 Overview

This notebook evaluates a **hybrid phishing detection workflow** using:

1. **Machine Learning (ML) Ensemble / Pretrained Model** – Fast first-line classification  
2. **Large Language Models (LLMs)** – Review of the most uncertain ML mistakes

---

## 🎯 Objective

Assess how well different LLMs perform at correcting **only the hardest ML mistakes**, without retraining or fully applying LLMs to all emails.  
The goal is to improve accuracy **efficiently**, focusing LLM attention where it's most needed.

---

## 📊 Workflow

1. **Load dataset** and clean duplicates, NaNs, and noisy content  
2. **ML predictions**: Use the pretrained model to classify all emails  
3. **Identify ML mistakes** and rank them by uncertainty (simulated confidence)  
4. **Select a limited subset** of the most uncertain mistakes for each LLM (e.g., 25 emails)  
5. **LLM evaluation**: Each LLM classifies the subset independently  
6. **Compare LLM performance** on ML mistakes using metrics (Accuracy, Precision, Recall, F1)

In [1]:
import pandas as pd
import numpy as np
import re
import os
import joblib
import time        
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from openai import OpenAI as OR_OpenAI
from dotenv import load_dotenv
from IPython.display import display

In [2]:
# Load .env into environment
load_dotenv()

# OpenRouter client (single API key to call multiple models)
OPENROUTER_KEY = os.getenv("OPENROUTER_API_KEY")
if not OPENROUTER_KEY:
    print("⚠️ OPENROUTER_API_KEY not found in .env. LLM calls will fail.")
    openrouter_client = None
else:
    try:
        openrouter_client = OR_OpenAI(api_key=OPENROUTER_KEY, base_url="https://openrouter.ai/api/v1")
        print("✅ OpenRouter client configured.")
    except Exception as e:
        openrouter_client = None
        print("⚠️ Failed to create OpenRouter client:", e)


MODEL_IDS = {
    "DeepSeek": "deepseek/deepseek-chat-v3.1",     
    "GPT": "openai/gpt-4.1",                       
    "Claude": "anthropic/claude-3.5-sonnet",            
}

# Thin wrapper — single function to call OpenRouter with chosen model name
def call_openrouter_model(prompt: str, model_id: str, temperature: float = 0.3, timeout: int = 30):
    """
    Calls OpenRouter chat completions using provided model_id.
    Returns (text, status) where status is "ok" or "error".
    """
    if openrouter_client is None:
        return ("❌ OpenRouter client not configured.", "error")
    try:
        resp = openrouter_client.chat.completions.create(
            model=model_id,
            messages=[{"role":"user","content":prompt}],
            temperature=temperature,
            max_tokens=800,
        )
        txt = resp.choices[0].message.content.strip()
        return (txt, "ok")
    except Exception as e:
        return (f"❌ OpenRouter request failed: {e}", "error")

✅ OpenRouter client configured.


In [3]:
dataset_path = "Dataset.csv"
df = pd.read_csv(dataset_path, engine="python", on_bad_lines="skip")
print(f"✅ Loaded Dataset.csv → {df.shape}")

# Standardize columns
if 'Email Text' in df.columns and 'Email Type' in df.columns:
    df = df[['Email Text', 'Email Type']]
    df.rename(columns={"Email Text": "email_text", "Email Type": "label"}, inplace=True)
else:
    raise ValueError("Dataset must have 'Email Text' and 'Email Type' columns")

# Drop NaN and duplicates
df = df.dropna(subset=['email_text', 'label'])
df = df.drop_duplicates(subset=['email_text'], keep='first')

# Clean text
def clean_text(text):
    text = re.sub(r'_+', ' ', text)
    text = re.sub(r'\b\d+\b', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

df['email_text'] = df['email_text'].astype(str).apply(clean_text)
df = df[df['email_text'] != '']

# Map labels to 0/1
def map_label(label):
    label_lower = str(label).lower()
    if 'phish' in label_lower or 'spam' in label_lower or 'malicious' in label_lower:
        return 1
    return 0

df['label'] = df['label'].apply(map_label).astype(int)

df = df.reset_index(drop=True)
print(f"\n✅ Cleaned dataset ready: {df.shape}")
print(df['label'].value_counts())

✅ Loaded Dataset.csv → (20698, 3)

✅ Cleaned dataset ready: (17534, 2)
label
0    10978
1     6556
Name: count, dtype: int64


In [4]:
# Paths
model_path = os.path.join("..", "ML pipeline", "model.pkl")
vectorizer_path = os.path.join("..", "ML pipeline", "tfidf_vectorizer.pkl")

# Load trained model and vectorizer
model = joblib.load(model_path)
vectorizer = joblib.load(vectorizer_path)
print("✅ Loaded ML model and TF-IDF vectorizer")

# Transform new email text
X_new_tfidf = vectorizer.transform(df['email_text'])

# True labels
y_true = df['label']

# Predict with ML model
ml_pred = model.predict(X_new_tfidf)
df['ml_pred'] = ml_pred

# ML performance
ml_accuracy = accuracy_score(y_true, ml_pred)
ml_precision = precision_score(y_true, ml_pred)
ml_recall = recall_score(y_true, ml_pred)
ml_f1 = f1_score(y_true, ml_pred)
ml_cm = confusion_matrix(y_true, ml_pred)

print("\n📊 ML-ONLY PERFORMANCE ON NEW DATASET")
print(f"Accuracy : {ml_accuracy:.4f}")
print(f"Precision: {ml_precision:.4f}")
print(f"Recall   : {ml_recall:.4f}")
print(f"F1 Score : {ml_f1:.4f}")
print("\nConfusion Matrix:")
print(ml_cm)

# Identify ML mistakes
ml_mistakes = df[df['ml_pred'] != df['label']].copy()
print(f"\n❌ ML mistakes found: {len(ml_mistakes)}")

✅ Loaded ML model and TF-IDF vectorizer

📊 ML-ONLY PERFORMANCE ON NEW DATASET
Accuracy : 0.9546
Precision: 0.8979
Recall   : 0.9913
F1 Score : 0.9423

Confusion Matrix:
[[10239   739]
 [   57  6499]]

❌ ML mistakes found: 796


In [5]:
print("\n" + "="*60)
print("⏱️ ML-ONLY PREDICTION TIME")
print("="*60)

start_ml = time.perf_counter()

X_new_tfidf = vectorizer.transform(df['email_text'])
ml_pred = model.predict(X_new_tfidf)

ml_time = time.perf_counter() - start_ml
ml_time_per_email = ml_time / len(df)

print(f"Total ML prediction time: {ml_time:.4f} seconds")
print(f"Average time per email:  {ml_time_per_email*1000:.2f} ms")


⏱️ ML-ONLY PREDICTION TIME
Total ML prediction time: 11.6612 seconds
Average time per email:  0.67 ms


In [6]:
# ✅ Identify ML mistakes
ml_mistakes = df[df['ml_pred'] != df['label']].copy()
# Simulate uncertainty for demo purposes
ml_mistakes['confidence'] = np.random.rand(len(ml_mistakes))

# Pick 25 most uncertain mistakes (lowest confidence)
uncertain_mistakes = ml_mistakes.nsmallest(25, 'confidence')
print(f"Selected {len(uncertain_mistakes)} most uncertain ML mistakes for LLM review")
display(uncertain_mistakes[['email_text', 'ml_pred', 'label', 'confidence']])

Selected 25 most uncertain ML mistakes for LLM review


,email_text,ml_pred,label,confidence
16067,"URL: http://www.newsisfree.com/click/- , , / D...",1,0,0.000720
10350,URL: http://scriptingnews.userland.com/backiss...,1,0,0.001402
4547,"list administrator, your authorization is requ...",1,0,0.001502
14550,"URL: http://www.newsisfree.com/click/- , , / D...",1,0,0.002451
15275,"URL: http://www.newsisfree.com/click/- , , / D...",1,0,0.004623
14309,Heh. Never mind the perfectly good desert in t...,1,0,0.004961
3800,"fwd : ever encountered in another author , and...",0,1,0.005825
3094,"URL: http://www.newsisfree.com/click/- , , / D...",1,0,0.006002
10699,URL: http://www.joelonsoftware.com/news/ .html...,1,0,0.006468
212,junk mail : books for linguists plurabelle boo...,0,1,0.007302


In [7]:
providers = list(MODEL_IDS.keys())
llm_time_total = {prov: 0.0 for prov in providers}
llm_call_count = {prov: 0 for prov in providers}

In [8]:
# Create a proper copy to avoid SettingWithCopyWarning
uncertain_mistakes = uncertain_mistakes.copy()

# Prepare columns for LLM predictions and statuses
for prov in providers:
    uncertain_mistakes[f'llm_{prov}_pred'] = None
    uncertain_mistakes[f'llm_{prov}_status'] = None

def llm_to_label(txt: str):
    """Parse LLM response to binary label."""
    txt = txt.lower()
    if "phish" in txt:
        return 1
    elif "safe" in txt:
        return 0
    return None

# Query each provider
print("\n🤖 Querying LLMs for classification...\n")
for idx, (i, row) in enumerate(uncertain_mistakes.iterrows(), 1):
    email_text = row['email_text']
    print(f"Processing email {idx}/{len(uncertain_mistakes)}...")
    
    for prov in providers:
        model_id = MODEL_IDS[prov]
        prompt = f"Classify this email as 'Phishing' or 'Safe':\n\n{email_text}"
        
        start_llm = time.perf_counter()

        resp, status = call_openrouter_model(prompt, model_id)
        
        elapsed = time.perf_counter() - start_llm
        llm_time_total[prov] += elapsed
        llm_call_count[prov] += 1

        pred = llm_to_label(resp) if status == "ok" else None
        
        uncertain_mistakes.at[i, f'llm_{prov}_pred'] = pred
        uncertain_mistakes.at[i, f'llm_{prov}_status'] = status
        
        print(f"  {prov}: status={status}, pred={pred}")
        time.sleep(0.5)  # Rate limiting
    print()  # Blank line between emails

print("✅ LLM classification complete!")


🤖 Querying LLMs for classification...

Processing email 1/25...
  DeepSeek: status=ok, pred=0
  GPT: status=ok, pred=1
  Claude: status=ok, pred=0

Processing email 2/25...
  DeepSeek: status=ok, pred=0
  GPT: status=ok, pred=1
  Claude: status=ok, pred=1

Processing email 3/25...
  DeepSeek: status=ok, pred=1
  GPT: status=ok, pred=1
  Claude: status=ok, pred=1

Processing email 4/25...
  DeepSeek: status=ok, pred=0
  GPT: status=ok, pred=1
  Claude: status=ok, pred=1

Processing email 5/25...
  DeepSeek: status=ok, pred=1
  GPT: status=ok, pred=1
  Claude: status=ok, pred=1

Processing email 6/25...
  DeepSeek: status=ok, pred=1
  GPT: status=ok, pred=1
  Claude: status=ok, pred=1

Processing email 7/25...
  DeepSeek: status=ok, pred=1
  GPT: status=ok, pred=1
  Claude: status=ok, pred=0

Processing email 8/25...
  DeepSeek: status=ok, pred=0
  GPT: status=ok, pred=1
  Claude: status=ok, pred=1

Processing email 9/25...
  DeepSeek: status=ok, pred=0
  GPT: status=ok, pred=1
  Claude

In [9]:
# Compute metrics only for emails that were sent to LLMs
print("\n📊 Accuracy of LLMs on 25 uncertain ML mistakes:")
metrics_summary = []

for prov in providers:
    col_pred = f'llm_{prov}_pred'
    # Only consider rows where LLM predicted
    mask = uncertain_mistakes[col_pred].notna()
    y_true_llm = uncertain_mistakes.loc[mask, 'label']
    y_pred_llm = uncertain_mistakes.loc[mask, col_pred].astype(int)
    
    acc = accuracy_score(y_true_llm, y_pred_llm)
    fixed = ((uncertain_mistakes['ml_pred'] != uncertain_mistakes['label']) & 
             (uncertain_mistakes[col_pred] == uncertain_mistakes['label'])).sum()
    
    metrics_summary.append({
        "Provider": prov,
        "Accuracy": acc,
        "ML Mistakes Fixed": fixed
    })
    print(f"{prov} - Accuracy: {acc:.4f}, ML mistakes fixed: {fixed}")

metrics_df = pd.DataFrame(metrics_summary)


📊 Accuracy of LLMs on 25 uncertain ML mistakes:
DeepSeek - Accuracy: 0.6000, ML mistakes fixed: 15
GPT - Accuracy: 0.1600, ML mistakes fixed: 4
Claude - Accuracy: 0.3200, ML mistakes fixed: 8


In [10]:
print("\n" + "="*60)
print("⏱️ LLM TIMING FOR 25 UNCERTAIN ML MISTAKES")
print("="*60)

for prov in providers:
    total_time = llm_time_total[prov]
    calls = llm_call_count[prov]

    if calls == 0:
        continue

    avg_time = total_time / calls

    print(f"\n{prov}")
    print(f"   Total LLM time: {total_time:.2f} seconds")
    print(f"   Calls made:     {calls}")
    print(f"   Avg per email:  {avg_time:.2f} seconds")


⏱️ LLM TIMING FOR 25 UNCERTAIN ML MISTAKES

DeepSeek
   Total LLM time: 405.73 seconds
   Calls made:     25
   Avg per email:  16.23 seconds

GPT
   Total LLM time: 120.07 seconds
   Calls made:     25
   Avg per email:  4.80 seconds

Claude
   Total LLM time: 133.35 seconds
   Calls made:     25
   Avg per email:  5.33 seconds
